# dsfb-swarm Colab Reproduction

`dsfb-swarm` is a reproducible demonstrator for deterministic spectral residual inference on swarm interaction networks. The notebook rebuilds the Rust crate from source, reruns the benchmark suite from scratch, loads the newest timestamped output bundle, shows the paper-facing figures inline, and then assembles a notebook-side PDF plus a zip archive for download.

## Technical overview

The crate evolves a time-varying swarm graph `G(t)` with weighted adjacency `A(t)`, degree `D(t)`, and Laplacian `L(t) = D(t) - A(t)`. At every step it computes the Laplacian spectrum, especially algebraic connectivity `lambda_2(t)`, and a monitored spectral stack `lambda_2(t) .. lambda_{m+1}(t)`.

The core diagnostic loop is deterministic:

- predict `hat lambda_k(t)` from previous spectral samples,
- compute residuals `r_k(t) = lambda_k(t) - hat lambda_k(t)`,
- compute residual drift and residual slew,
- compute residual envelopes and anomaly certificates,
- optionally compare mode shapes through eigenvector residuals with sign ambiguity handling,
- attenuate interactions through trust-gated effective weights `a_ij(t) = T_ij(t) * tilde a_ij(t)`.

The benchmark suite evaluates four scenarios aligned with the paper:

1. nominal coordination,
2. gradual edge degradation,
3. adversarial agent influence,
4. communication loss and fragmentation.

The important outputs are not just plots of `lambda_2(t)`. The crate also exports residuals, drift, slew, envelopes, trust trajectories, baseline comparisons, scaling metrics, and noise-stress metrics so the paper’s structural claims can be read as empirical diagnostics rather than as labels pasted onto a generic simulation.


## Notebook flow

1. Bootstrap Python and Rust tooling.
2. Locate the DSFB repository or clone it if needed.
3. Build `crates/dsfb-swarm` from source.
4. Run the benchmark suite from scratch.
5. Locate the newest `output-dsfb-swarm/<timestamp>/` directory.
6. Load CSV and JSON artifacts.
7. Display the generated figures inline.
8. Assemble a notebook-side PDF report.
9. Create a zip archive containing the full artifact bundle.


In [ ]:
import json
import os
import shutil
import subprocess
import sys
import textwrap
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import FileLink, Image, Markdown, display
from matplotlib.backends.backend_pdf import PdfPages


def run_command(cmd, cwd=None):
    print("$", " ".join(str(part) for part in cmd))
    subprocess.run(cmd, check=True, cwd=cwd)


subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "matplotlib"], check=True)

if shutil.which("rustc") is None:
    run_command(["bash", "-lc", "curl https://sh.rustup.rs -sSf | sh -s -- -y"])

cargo_bin = Path.home() / ".cargo" / "bin"
os.environ["PATH"] = f"{cargo_bin}:{os.environ['PATH']}"
run_command(["rustc", "--version"])
run_command(["cargo", "--version"])


## Locate or clone the repository

The notebook first searches the local Colab filesystem for a repository containing `crates/dsfb-swarm/Cargo.toml`. If nothing suitable is found, it clones the repository into `/content/dsfb`.

By default the notebook clones `https://github.com/infinityabundance/dsfb.git` when it cannot find a local repository. You can still override that with `DSFB_REPO_URL`, `DSFB_REPO_REF`, or `DSFB_REPO_ROOT`.


In [ ]:
REPO_ROOT_OVERRIDE = os.environ.get("DSFB_REPO_ROOT", "").strip()
DEFAULT_REPO_URL = os.environ.get("DSFB_REPO_URL", "https://github.com/infinityabundance/dsfb.git").strip()
REPO_REF = os.environ.get("DSFB_REPO_REF", "main").strip() or "main"
CLONE_DIR = Path("/content/dsfb")


def has_repo_root(path: Path) -> bool:
    return (path / "crates" / "dsfb-swarm" / "Cargo.toml").exists()


def find_repo_root() -> Path | None:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([Path("/content"), Path("/content/dsfb"), Path("/content/repo")])
    if Path("/content").exists():
        for child in sorted(Path("/content").iterdir()):
            candidates.append(child)
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except FileNotFoundError:
            continue
        if resolved in seen:
            continue
        seen.add(resolved)
        if has_repo_root(resolved):
            return resolved
    return None


repo_root = Path(REPO_ROOT_OVERRIDE).resolve() if REPO_ROOT_OVERRIDE else None
if repo_root is None:
    if CLONE_DIR.exists():
        shutil.rmtree(CLONE_DIR)
    run_command(["git", "clone", "--depth", "1", "--branch", REPO_REF, DEFAULT_REPO_URL, str(CLONE_DIR)])
    repo_root = CLONE_DIR

crate_dir = repo_root / "crates" / "dsfb-swarm"
notebook_path = crate_dir / "notebooks" / "dsfb_swarm_colab.ipynb"
manifest_path = crate_dir / "Cargo.toml"
output_root = crate_dir / "output-dsfb-swarm"
print("repo_root:", repo_root)
print("crate_dir:", crate_dir)
print("notebook_path:", notebook_path)
print("output_root:", output_root)
assert manifest_path.exists(), manifest_path


## Build and run from scratch

The benchmark suite below rebuilds the crate and reruns all four scenarios using the crate-local output root. That preserves the self-contained execution model of `dsfb-swarm` while still producing a fresh timestamped artifact bundle every time.


In [ ]:
if output_root.exists():
    shutil.rmtree(output_root)
output_root.mkdir(parents=True, exist_ok=True)

run_command(["cargo", "build", "--manifest-path", str(manifest_path), "--release"])
run_command([
    "cargo",
    "run",
    "--manifest-path",
    str(manifest_path),
    "--release",
    "--",
    "benchmark",
    "--all-scenarios",
    "--sizes",
    "20,50,100",
    "--noise",
    "0.01,0.05,0.10",
    "--steps",
    "120",
    "--modes",
    "4",
    "--mode-shapes",
    "--output-root",
    str(output_root),
])

generated_runs = sorted(path for path in output_root.iterdir() if path.is_dir())
if len(generated_runs) != 1:
    raise RuntimeError(f"Expected exactly one fresh benchmark run under {output_root}, found {len(generated_runs)}")
latest_benchmark_run_dir = generated_runs[0]
print("benchmark run_dir:", latest_benchmark_run_dir)


In [ ]:
run_dir = latest_benchmark_run_dir
figure_dir = run_dir / "figures"
report_dir = run_dir / "report"
print("latest run_dir:", run_dir)

required_run_artifacts = [
    "scenarios_summary.csv",
    "benchmark_summary.csv",
    "hero_benchmark_summary.csv",
    "time_series.csv",
    "manifest.json",
]
missing_run_artifacts = [name for name in required_run_artifacts if not (run_dir / name).exists()]
if missing_run_artifacts:
    raise RuntimeError(
        "Benchmark run did not generate all required empirical artifacts: " + ", ".join(missing_run_artifacts)
    )

summary = pd.read_csv(run_dir / "scenarios_summary.csv")
benchmark = pd.read_csv(run_dir / "benchmark_summary.csv")
hero_benchmark = pd.read_csv(run_dir / "hero_benchmark_summary.csv")
time_series = pd.read_csv(run_dir / "time_series.csv")
manifest = json.loads((run_dir / "manifest.json").read_text())

display(Markdown("### Scenario summary"))
display(summary)
display(Markdown("### Benchmark preview"))
display(benchmark.head(16))
display(Markdown("### Hero benchmark rows"))
display(hero_benchmark)
display(Markdown("### Manifest keys"))
display(pd.DataFrame({"key": list(manifest.keys())}))


In [ ]:
figure_names = [
    "lambda2_timeseries.png",
    "residual_timeseries.png",
    "drift_slew.png",
    "trust_evolution.png",
    "baseline_comparison.png",
    "scaling_curves.png",
    "noise_stress_curves.png",
    "multimode_comparison.png",
    "topology_snapshots.png",
    "hero_leadtime_comparison.png",
    "hero_benchmark_table.png",
]

missing_figures = [name for name in figure_names if not (figure_dir / name).exists()]
if missing_figures:
    raise RuntimeError(
        "Benchmark run did not generate all required figures: " + ", ".join(missing_figures)
    )

for name in figure_names:
    display(Markdown(f"### {name}"))
    display(Image(filename=str(figure_dir / name)))


In [ ]:
pdf_path = report_dir / "dsfb_swarm_colab_report.pdf"
zip_path = report_dir / "dsfb_swarm_artifacts.zip"
manifest_json_path = report_dir / "colab_artifact_manifest.json"
notebook_copy_path = report_dir / "dsfb_swarm_colab.ipynb"

shutil.copy2(notebook_path, notebook_copy_path)

def format_pdf_value(value):
    if pd.isna(value):
        return "n/a"
    if isinstance(value, bool):
        return str(value)
    if isinstance(value, (int, float)):
        numeric = float(value)
        if numeric.is_integer() and abs(numeric) >= 1.0:
            return str(int(numeric))
        return f"{numeric:.4f}"
    return str(value)

def render_table_pages(pdf, title, frame, key_columns, extra_columns_per_page=6, rows_per_page=10):
    formatted = frame.copy()
    for column in formatted.columns:
        formatted[column] = formatted[column].map(format_pdf_value)

    key_columns = [column for column in key_columns if column in formatted.columns]
    variable_columns = [column for column in formatted.columns if column not in key_columns]
    column_groups = [
        key_columns + variable_columns[index:index + extra_columns_per_page]
        for index in range(0, len(variable_columns), extra_columns_per_page)
    ] or [list(formatted.columns)]

    for group_index, columns in enumerate(column_groups):
        grouped = formatted[columns]
        for start in range(0, len(grouped), rows_per_page):
            chunk = grouped.iloc[start:start + rows_per_page]
            fig, ax = plt.subplots(figsize=(11.69, 8.27))
            ax.axis("off")

            notes = []
            if len(column_groups) > 1:
                notes.append(f"cols {group_index + 1}/{len(column_groups)}")
            if len(grouped) > rows_per_page:
                notes.append(f"rows {start + 1}-{min(start + rows_per_page, len(grouped))} of {len(grouped)}")
            page_title = title if not notes else f"{title} ({'; '.join(notes)})"
            ax.set_title(page_title, fontsize=13, pad=10)

            table = ax.table(
                cellText=chunk.values,
                colLabels=list(chunk.columns),
                cellLoc="center",
                loc="center",
                bbox=[0.02, 0.03, 0.96, 0.86],
            )
            table.auto_set_font_size(False)
            font_size = max(4.6, 7.4 - 0.18 * len(chunk.columns))
            table.set_fontsize(font_size)
            table.scale(1.0, 1.1)

            key_width_total = 0.30 if key_columns else 0.0
            key_width = key_width_total / max(1, len(key_columns)) if key_columns else 0.0
            other_width = (0.96 - key_width_total) / max(1, len(chunk.columns) - len(key_columns))
            for col_index, column in enumerate(chunk.columns):
                width = key_width if column in key_columns else other_width
                for row_index in range(len(chunk) + 1):
                    cell = table[(row_index, col_index)]
                    cell.set_width(width)
                    cell.get_text().set_wrap(True)
                    if row_index == 0:
                        cell.set_facecolor("#e5e7eb")
                        cell.get_text().set_fontweight("bold")

            fig.tight_layout(pad=0.3)
            pdf.savefig(fig, bbox_inches="tight", pad_inches=0.15)
            plt.close(fig)

with PdfPages(pdf_path) as pdf:
    fig, ax = plt.subplots(figsize=(11.69, 8.27))
    ax.axis("off")
    intro = textwrap.dedent(
        f"""
        dsfb-swarm Colab report

        Run directory: {run_dir}

        This notebook rebuilt the crate, reran the benchmark suite, loaded the newest timestamped output bundle,
        and assembled this PDF from the generated artifacts. The report focuses on deterministic spectral prediction,
        residual-centered diagnosis, drift and slew monitoring, trust-gated attenuation, scaling, noise stress, and
        the difference between scalar lambda_2 monitoring and stacked multi-mode monitoring.
        """
    )
    ax.text(0.03, 0.97, intro, va="top", ha="left", fontsize=10.5, family="monospace", wrap=True)
    fig.tight_layout(pad=0.4)
    pdf.savefig(fig, bbox_inches="tight", pad_inches=0.2)
    plt.close(fig)

    render_table_pages(
        pdf,
        "Scenario summary",
        summary,
        ["scenario", "scenario_kind"],
        extra_columns_per_page=6,
        rows_per_page=8,
    )
    render_table_pages(
        pdf,
        "Benchmark summary",
        benchmark.head(20),
        ["scenario", "scenario_kind", "agents", "noise_level"],
        extra_columns_per_page=6,
        rows_per_page=10,
    )
    render_table_pages(
        pdf,
        "Hero benchmark rows",
        hero_benchmark,
        ["scenario", "agents", "noise_level"],
        extra_columns_per_page=6,
        rows_per_page=8,
    )

    for name in figure_names:
        img = plt.imread(figure_dir / name)
        fig, ax = plt.subplots(figsize=(11.69, 8.27))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(name, fontsize=12, pad=8)
        fig.tight_layout(pad=0.3)
        pdf.savefig(fig, bbox_inches="tight", pad_inches=0.15)
        plt.close(fig)

files_to_zip = sorted(
    path for path in run_dir.rglob("*")
    if path.is_file() and path != zip_path
)
artifact_manifest = {
    "run_dir": str(run_dir),
    "figure_dir": str(figure_dir),
    "report_dir": str(report_dir),
    "notebook_pdf": pdf_path.name,
    "artifact_zip": zip_path.name,
    "notebook_copy": notebook_copy_path.name,
    "figure_files": figure_names,
    "file_count": len(files_to_zip),
}
manifest_json_path.write_text(json.dumps(artifact_manifest, indent=2))
files_to_zip = sorted(
    path for path in run_dir.rglob("*")
    if path.is_file() and path != zip_path
)
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in files_to_zip:
        archive.write(path, path.relative_to(run_dir))

print("Notebook PDF:", pdf_path)
print("Artifact ZIP:", zip_path)
print("Artifact manifest:", manifest_json_path)
print("Notebook copy:", notebook_copy_path)
print("Files packaged:", len(files_to_zip))

display(Markdown("### Download links"))
display(FileLink(str(pdf_path), result_html_prefix="Notebook PDF: "))
display(FileLink(str(zip_path), result_html_prefix="Artifact ZIP: "))
display(FileLink(str(manifest_json_path), result_html_prefix="Artifact manifest: "))


## Artifact notes

- The Rust crate already writes its own Markdown and compact PDF report into `report/`.
- This notebook adds `report/dsfb_swarm_colab_report.pdf`, a notebook-generated PDF of the Colab results.
- This notebook writes `report/colab_artifact_manifest.json` so the bundle contents are explicit and machine-readable.
- This notebook copies `notebooks/dsfb_swarm_colab.ipynb` into the run report folder so the executed workflow is packaged with the outputs.
- The zip archive `report/dsfb_swarm_artifacts.zip` contains the run’s CSVs, JSON files, PNG figures, Markdown report, Rust PDF report, notebook PDF, notebook copy, and artifact manifest.
- The final cell also displays direct download links for the PDF, zip archive, and manifest.
